In [1]:
import numpy as np
import pandas as pd
import pygad
import time
import inventory_model as im

N, W = im.N, im.W
weekly_fc = im.weekly_fc
DEFAULTS = dict(budget_factor=0.95, price=18.0, hold=0.1)

def set_params(budget_factor, price, hold):
    im.BUDGET_FACTOR = budget_factor
    im.PRICE = price
    im.HOLD = hold
    im.BUDGET = budget_factor * im.COST * weekly_fc.sum()

def decode(genes):
    return np.clip(np.array(genes), 0, 1.5).reshape(N, W) * weekly_fc

def fitness_func(ga_instance, solution, solution_idx):
    return im.fitness(decode(solution))

def run_case(name, budget_factor=0.95, price=18.0, hold=0.1, seed=42, generations=300):
    set_params(budget_factor, price, hold)
    b = min(budget_factor, 0.95)
    baseline = b * weekly_fc

    rng = np.random.default_rng(seed)
    pop = rng.uniform(0, 1.5, size=(80, N * W))
    pop[:60] = np.clip(b + rng.normal(0, 0.05, size=(60, N * W)), 0, 1.5)
    pop[0, :] = b

    ga = pygad.GA(
        num_generations=generations,
        num_parents_mating=40,
        initial_population=pop,
        fitness_func=fitness_func,
        gene_type=float,
        parent_selection_type="tournament",
        K_tournament=3,
        crossover_type="uniform",
        mutation_type="random",
        mutation_percent_genes=15,
        mutation_by_replacement=False,
        random_mutation_min_val=-0.03,
        random_mutation_max_val=0.03,
        keep_elitism=2,
        random_seed=seed,
    )
    ga.run()
    sol, _, _ = ga.best_solution()
    plan = decode(sol)

    base_v = im.simulate_profit(baseline, im.val_scen).mean()
    ga_v = im.simulate_profit(plan, im.val_scen).mean()
    pen = im.penalty(plan)

    set_params(**DEFAULTS)   # reset to the default assumptions
    return {
        "case": name,
        "baseline_val": round(base_v),
        "ga_val": round(ga_v),
        "gain_%": round((ga_v / base_v - 1) * 100, 2),
        "penalty": round(pen),
    }

In [2]:
cases = [
    ("Tight budget (0.90)",     dict(budget_factor=0.90)),
    ("Looser budget (1.00)",    dict(budget_factor=1.00)),
    ("Generous budget (1.10)",  dict(budget_factor=1.10)),
    ("Low margin (price 14)",   dict(price=14.0)),
    ("High margin (price 25)",  dict(price=25.0)),
    ("Costly storage (hold 0.5)", dict(hold=0.5)),
]

rows = []
for name, kwargs in cases:
    start = time.time()
    rows.append(run_case(name, **kwargs))
    print(f"{name} done in {(time.time() - start) / 60:.1f} min | gain {rows[-1]['gain_%']}%")

sens = pd.DataFrame(rows)
sens.to_csv("sensitivity_results.csv", index=False)

Tight budget (0.90) done in 3.3 min | gain 0.06%
Looser budget (1.00) done in 2.5 min | gain 2.55%
Generous budget (1.10) done in 2.4 min | gain 2.55%
Low margin (price 14) done in 2.3 min | gain 0.7%
High margin (price 25) done in 2.3 min | gain 0.32%
Costly storage (hold 0.5) done in 2.2 min | gain 0.37%


In [3]:
print(sens.to_string(index=False))

                     case  baseline_val  ga_val  gain_%  penalty
      Tight budget (0.90)        716434  716837    0.06        0
     Looser budget (1.00)        749270  768412    2.55        0
   Generous budget (1.10)        749270  768412    2.55        0
    Low margin (price 14)        359160  361669    0.70        0
   High margin (price 25)       1431963 1436586    0.32       47
Costly storage (hold 0.5)        638462  640809    0.37        0


In [4]:
import numpy as np
import inventory_model as im

fair = 1.00 * im.weekly_fc
v = im.simulate_profit(fair, im.val_scen).mean()
print(f"Baseline (order 100% of forecast): validation mean {v:,.0f}")
print(f"GA (budget 1.00, validation mean 768,412) vs this baseline: {(768412 / v - 1):.2%}")

Baseline (order 100% of forecast): validation mean 760,522
GA (budget 1.00, validation mean 768,412) vs this baseline: 1.04%
